In [0]:
# Project         : Procurement Analytics using Databricks & Power BI
# Layer           : Gold
# Notebook        : Gold_Fact_Goods_Receipts
# Source          : Goods_Receipts.csv
# Target          : procurement.gold.fact_goods_receipts

#
# Author          : V R Mutyala
# Created Date    : 21-Jul-2026
# Last Modified   : 21-Jul-2026
#
# Description
# -----------
# This notebook created the Fact_Goods_Receipts table in the gold layer.
#
# ==============================================================================
# Business Objective
# ==============================================================================
# Create the Fact_Goods_Receipts Fact table from the Silver layer.
# ==============================================================================

In [0]:
%run ../01_Config/Config

In [0]:
%run ../05_Helper_Functions/Helper_functions

In [0]:
# ============================================================
# Read Silver Goods_Receipts Table
# ============================================================

Silver_Goods_Receipts_df = read_delta(SILVER_GOODS_RECEIPTS)

preview(Silver_Goods_Receipts_df,"Silver Goods_Receipts")

In [0]:
Silver_Purchase_Orders_df = read_delta(SILVER_PURCHASE_ORDERS)
Silver_Employees_df = read_delta(SILVER_EMPLOYEES)

In [0]:
from pyspark.sql import functions as F

# Join Goods Receipts with Purchase Orders and Employees
gold_Goods_Receipts_base_df = (
    Silver_Goods_Receipts_df.alias("gr")

    # Join with Purchase Orders using po_id
    .join(
        Silver_Purchase_Orders_df.alias("po"),
        F.col("gr.po_id") == F.col("po.po_id"),
        "left"
    )

    # Join with Employees using received_by and employee_id
    .join(
        Silver_Employees_df.alias("emp"),
        F.col("gr.received_by") == F.col("emp.employee_id"),
        "left"
    )

    # Select columns available in your tables
    .select(
        # Goods Receipts
        F.col("gr.grn_id"),
        F.col("gr.grn_date"),
        F.col("gr.po_id"),
        F.col("gr.quantity_ordered_ref"),
        F.col("gr.quantity_received"),
        F.col("gr.receipt_status"),
        F.col("gr.received_by"),

        # Purchase Orders
        F.col("po.pr_id"),
        F.col("po.supplier_id"),
        F.col("po.product_id"),
        F.col("po.po_date"),
        F.col("po.quantity_ordered"),
        F.col("po.unit_price"),
        F.col("po.total_amount"),
        F.col("po.currency"),
        F.col("po.po_status"),
        F.col("po.is_duplicate_po"),

        # Employees
        F.col("emp.employee_name").alias("received_by_name"),
        F.col("emp.department_id").alias("receiver_department_id"),
        F.col("emp.job_title").alias("receiver_job_title"),
        F.col("emp.is_active").alias("receiver_is_active")
    )
)

In [0]:
# 1. Source row counts
print("Silver Goods Receipts:", Silver_Goods_Receipts_df.count())
print("Silver Purchase Orders:", Silver_Purchase_Orders_df.count())
print("Silver Employees:", Silver_Employees_df.count())

# 2. Gold base row count
print("Gold Goods Receipts:", gold_Goods_Receipts_base_df.count())

# 3. Check GRN uniqueness
print(
    "Distinct GRN IDs:",
    gold_Goods_Receipts_base_df.select("grn_id").distinct().count()
)

# 4. Check duplicate GRN IDs
gold_Goods_Receipts_base_df.groupBy("grn_id").count().filter(F.col("count") > 1).show(truncate=False)

In [0]:
po_received_summary_df = (
    gold_Goods_Receipts_base_df
    .groupBy("po_id")
    .agg(
        F.sum("quantity_received").alias("total_quantity_received"),
        F.max("quantity_ordered").alias("quantity_ordered")
    )
)
display(po_received_summary_df)

In [0]:
po_received_summary_df = (
    po_received_summary_df
    .withColumn(
        "quantity_variance",
        F.col("total_quantity_received") -
        F.col("quantity_ordered")
    )
)

In [0]:
display(po_received_summary_df)

In [0]:
po_received_summary_df = (
    po_received_summary_df
    .withColumn(
        "quantity_reconciliation_status",
        F.when(
            F.col("quantity_variance") < 0,
            "Short Received"
        )
        .when(
            F.col("quantity_variance") == 0,
            "Fully Received"
        )
        .otherwise("Excess Received")
    )
)

In [0]:
display(po_received_summary_df)

In [0]:
po_received_summary_df.select(
    "po_id",
    "quantity_ordered",
    "total_quantity_received",
    "quantity_variance",
    "quantity_reconciliation_status"
).show(20, truncate=False)

In [0]:
# ============================================================
# Add Gold Metadata
# ============================================================

gold_fact_Goods_Receipts_df = (
   gold_Goods_Receipts_base_df.withColumn("gold_load_timestamp", F.current_timestamp())
)

In [0]:
# ============================================================
# Write Gold Delta Table
# ============================================================

write_delta(
    df=gold_fact_Goods_Receipts_df,
    table_name=GOLD_FACT_GOODS_RECEIPTS
)

In [0]:
# ============================================================
# Validate Gold Delta Table
# ============================================================

validate_table(GOLD_FACT_GOODS_RECEIPTS)

In [0]:
gr_received_summary_df = (
    Silver_Goods_Receipts_df
    .groupBy("po_id")
    .agg(
        F.sum("quantity_received").alias("total_quantity_received"),
        F.countDistinct("grn_id").alias("total_grns")
    )
)

In [0]:
# ============================================================
# Gold Layer Summary
# ============================================================

print("=" * 60)
print("Gold Payments Load Completed Successfully")
print("=" * 60)

print(f"Silver Records : {Silver_Goods_Receipts_df.count():,}")
print(f"Gold Records   : {Gold_Goods_Receipts.ipynb.count():,}")